In [1]:
# =============================================================================
# IMPORTS
# =============================================================================
import os
import platform
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils import resample
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, roc_curve, precision_recall_curve, auc
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import hashlib
import random

# =============================================================================
# TPU/GPU DETECTION
# =============================================================================
print("\n" + "="*80)
print("HARDWARE DETECTION")
print("="*80)
print(f"PyTorch: {torch.__version__}")
print(f"Platform: {platform.platform()}")

# TPU setup
TPU_AVAILABLE = False
xm = None
xmp = None

try:
    import torch_xla
    import torch_xla.core.xla_model as xla_model
    TPU_AVAILABLE = True
    xm = xla_model
    print(f"✅ TPU Available: torch_xla {torch_xla.__version__}")
    try:
        import torch_xla.distributed.xla_multiprocessing as xla_mp
        xmp = xla_mp
    except ImportError:
        pass
except ImportError:
    print("⚠️ TPU not available (torch_xla not installed)")
    print("   For Kaggle TPU: Runtime → Change runtime type → TPU")

# CUDA setup
if torch.cuda.is_available():
    print(f"\n✅ CUDA Available")
    print(f"   GPU Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"   GPU {i}: {props.name} ({props.total_memory / 1e9:.2f} GB)")
else:
    print("\n⚠️ CUDA not available")

print("="*80 + "\n")

# Device selection
USE_XLA = False
DEVICE = None

if TPU_AVAILABLE and xm is not None:
    USE_XLA = True
    # Use torch_xla.device() instead of deprecated xm.xla_device()
    try:
        import torch_xla
        DEVICE = torch_xla.device()
    except:
        # Fallback to old method if new method fails
        DEVICE = xm.xla_device()
    print(f"🎯 Using TPU: {DEVICE}")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"🎯 Using CUDA: {DEVICE}")
else:
    DEVICE = torch.device("cpu")
    print(f"🎯 Using CPU: {DEVICE}")

print("")



# =============================================================================
# CONFIGURATION
# =============================================================================

# Dataset paths - CIC-DDoS2019
TRAIN_DATA_DIR = "/kaggle/input/cic-ddos2019-30gb-full-dataset-csv-files/01-12"
TEST_DATA_DIR = "/kaggle/input/cic-ddos2019-30gb-full-dataset-csv-files/03-11"

# Training files (Day 1 - 01-12)
TRAIN_FILES = [
    "TFTP.csv",
    "DrDoS_LDAP.csv",
    "DrDoS_MSSQL.csv",
    "DrDoS_NetBIOS.csv",
    "DrDoS_NTP.csv",
    "DrDoS_SNMP.csv",
    "DrDoS_SSDP.csv",
    "DrDoS_UDP.csv",
    "Syn.csv",
    "DrDoS_DNS.csv",
    "UDPLag.csv"
]

# Test files (Day 2 - 03-11)
TEST_FILES = [
    "LDAP.csv",
    "MSSQL.csv",
    "NetBIOS.csv",
    "Portmap.csv",
    "Syn.csv"
    # Note: UDP.csv and UDPLag.csv can be added if fixed
]

# Downsampling configuration
MULT = 5  # Multiplier to control anomaly class size (attack flows = benign flows * MULT)

# Toggle between test mode and full data mode
USE_FULL_DATA = True  # Set to True for full dataset, False for quick testing
NUM_SAMPLES = None if USE_FULL_DATA else 5000  # None = unlimited, 5000 = quick test

# Output
OUT_DIR = "/kaggle/working/models"
PLOT_DIR = "/kaggle/working/plots"
PROCESSED_DATA_DIR = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

# Set seed for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)


# =============================================================================
# DATA LOADING AND PREPROCESSING FUNCTIONS
# =============================================================================

def string2numeric_hash(text):
    """Convert string to numeric hash for timestamp processing."""
    return int(hashlib.md5(text.encode()).hexdigest()[:8], 16)


def load_file(path, num_samples=NUM_SAMPLES, mult=MULT):
    """
    Load and downsample a single CSV file.
    
    Balances benign and anomalous flows by downsampling the majority class.
    Anomalous flows can be 'mult' times larger than benign flows to prevent
    excessive information loss while reducing class imbalance.
    
    Args:
        path: Path to CSV file
        num_samples: Maximum number of rows to read (None for unlimited)
        mult: Multiplier for target anomaly class size
        
    Returns:
        Tuple of (flows_ok, flows_ddos_reduced): Two separate DataFrames for benign and attack flows
    """
    # Count total lines in the file (excluding header)
    total_lines = sum(1 for _ in open(path, encoding='utf-8', errors='ignore')) - 1
    
    # Ensure at most num_samples rows are read randomly
    if num_samples and total_lines > num_samples:
        skip_rows = sorted(random.sample(range(1, total_lines + 1), total_lines - num_samples))
    else:
        skip_rows = None
    
    # Read CSV file
    data = pd.read_csv(path, skiprows=skip_rows, sep=',', low_memory=False, encoding='utf-8', encoding_errors='ignore')
    
    # Separate benign and anomalous flows
    is_benign = data[' Label'] == 'BENIGN'
    flows_ok = data[is_benign]
    flows_ddos_full = data[~is_benign]
    
    # Calculate target size for anomalous data
    sizeDownSample = len(flows_ok) * mult
    
    # Downsample majority class if necessary
    if sizeDownSample < len(flows_ddos_full):
        flows_ddos_reduced = resample(
            flows_ddos_full,
            replace=False,
            n_samples=sizeDownSample,
            random_state=SEED
        )
    else:
        flows_ddos_reduced = flows_ddos_full
    
    # Return two separate DataFrames (DO NOT concatenate here)
    return flows_ok, flows_ddos_reduced


def load_huge_file(path, num_samples=NUM_SAMPLES, mult=MULT, chunksize=500000):
    """
    Load and downsample a large CSV file in chunks.
    
    For very large files, reads and processes data in chunks to manage memory.
    Each chunk is processed separately, then combined at the end.
    
    Args:
        path: Path to CSV file
        num_samples: Maximum number of rows to read (None for unlimited)
        mult: Multiplier for target anomaly class size
        chunksize: Number of rows per chunk
        
    Returns:
        Tuple of (flows_ok, flows_ddos): Two separate DataFrames for benign and attack flows
    """
    # Count total lines
    total_lines = sum(1 for _ in open(path, encoding='utf-8', errors='ignore')) - 1
    
    # Determine which rows to skip
    if num_samples and total_lines > num_samples:
        skip_rows = sorted(random.sample(range(1, total_lines + 1), total_lines - num_samples))
    else:
        skip_rows = None
    
    # Read in chunks
    df_chunk = pd.read_csv(path, skiprows=skip_rows, chunksize=chunksize, low_memory=False, encoding='utf-8', encoding_errors='ignore')
    
    chunk_list_ok = []
    chunk_list_ddos = []
    
    # Process each chunk
    for chunk in df_chunk:
        is_benign = chunk[' Label'] == 'BENIGN'
        flows_ok = chunk[is_benign]
        flows_ddos_full = chunk[~is_benign]
        
        # Downsample anomalous flows in this chunk
        if (len(flows_ok) * mult) < len(flows_ddos_full):
            sizeDownSample = len(flows_ok) * mult
            flows_ddos_reduced = resample(
                flows_ddos_full,
                replace=False,
                n_samples=sizeDownSample,
                random_state=SEED
            )
        else:
            flows_ddos_reduced = flows_ddos_full
        
        chunk_list_ok.append(flows_ok)
        chunk_list_ddos.append(flows_ddos_reduced)
    
    # Combine all chunks but keep benign and attack separate
    flows_ok = pd.concat(chunk_list_ok)
    flows_ddos = pd.concat(chunk_list_ddos)
    
    # Return two separate DataFrames (DO NOT concatenate here)
    return flows_ok, flows_ddos


def preprocess_dataframe(df, dataset_type='train'):
    """
    Preprocess CIC-DDoS2019 dataset.
    
    Steps:
    1. Replace infinite values with 0
    2. Convert numerical columns safely
    3. Map attack labels to binary (0=BENIGN, 1=ATTACK)
    4. Hash timestamps to numeric values (preserves temporal information)
    5. Drop unnecessary columns
    
    Args:
        df: Raw DataFrame
        dataset_type: 'train' or 'test' (affects label mapping)
        
    Returns:
        Preprocessed DataFrame
    """
    # Replace infinite values
    df = df.replace(['Infinity', np.inf, -np.inf], 0)
    
    # Convert numerical columns safely
    if ' Flow Packets/s' in df.columns:
        df[' Flow Packets/s'] = pd.to_numeric(df[' Flow Packets/s'], errors='coerce').fillna(0)
    if 'Flow Bytes/s' in df.columns:
        df['Flow Bytes/s'] = pd.to_numeric(df['Flow Bytes/s'], errors='coerce').fillna(0)
    
    # Map labels to binary classification
    if dataset_type == 'train':
        # Day 1 attack types
        label_map = {
            'BENIGN': 0,
            'DrDoS_DNS': 1,
            'DrDoS_LDAP': 1,
            'DrDoS_MSSQL': 1,
            'DrDoS_NTP': 1,
            'DrDoS_NetBIOS': 1,
            'DrDoS_SNMP': 1,
            'DrDoS_SSDP': 1,
            'DrDoS_UDP': 1,
            'Syn': 1,
            'TFTP': 1,
            'UDP-lag': 1,
            'WebDDoS': 1
        }
    else:  # test
        # Day 2 attack types
        label_map = {
            'BENIGN': 0,
            'LDAP': 1,
            'NetBIOS': 1,
            'MSSQL': 1,
            'Portmap': 1,
            'Syn': 1
        }
    
    df[' Label'] = df[' Label'].replace(label_map).astype(int)
    
    # Hash Timestamp to numeric value (like compare.md)
    # Format: "DD/MM/YYYY HH:MM:SS.microseconds" → extract time → hash
    if ' Timestamp' in df.columns:
        # Split timestamp: "DD/MM/YYYY HH:MM:SS" → keep only time part
        colunaTime = df[' Timestamp'].str.split(' ', n=1, expand=True)
        if colunaTime.shape[1] == 2:
            # Get time part (HH:MM:SS.microseconds)
            time_part = colunaTime[1].str.split('.', n=1, expand=True)[0]  # Remove microseconds
            # Hash the time string to numeric
            df[' Timestamp'] = time_part.apply(lambda x: string2numeric_hash(str(x)) if pd.notna(x) else 0)
        else:
            # Fallback: if format is different, just hash the whole thing
            df[' Timestamp'] = df[' Timestamp'].apply(lambda x: string2numeric_hash(str(x)) if pd.notna(x) else 0)
    
    # Drop unnecessary columns (but keep hashed Timestamp)
    columns_to_drop = []
    for col in [' Source IP', ' Destination IP', 'Flow ID', 'SimillarHTTP', 'Unnamed: 0']:
        if col in df.columns:
            columns_to_drop.append(col)
    
    if columns_to_drop:
        df.drop(columns=columns_to_drop, inplace=True)
    
    return df


def normalize_data(X_train, X_test):
    """
    Normalize data between -1 and 1 using MinMaxScaler.
    
    Args:
        X_train: Training features
        X_test: Test features
        
    Returns:
        Normalized X_train, X_test
    """
    scaler = MinMaxScaler(feature_range=(-1, 1)).fit(X_train)
    X_train = scaler.transform(X_train)
    X_test = scaler.transform(X_test)
    return X_train, X_test


def train_test_split_custom(samples, test_size=0.33):
    """
    Split samples into train and test sets.
    
    Args:
        samples: DataFrame with features and label
        test_size: Fraction of data for testing
        
    Returns:
        X_train, X_test, y_train, y_test
    """
    X = samples.iloc[:, :-1]  # All columns except last
    y = samples.iloc[:, -1]   # Last column (label)
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=SEED
    )
    
    return X_train, X_test, y_train, y_test


def test_normal_atk(y_test, y_pred):
    """
    Calculate detection rates for normal and attack flows separately.
    
    Args:
        y_test: True labels
        y_pred: Predicted labels
        
    Returns:
        normal_detect_rate, atk_detect_rate
    """
    df = pd.DataFrame({'y_test': y_test, 'y_pred': y_pred})
    
    normal = df[df['y_test'] == 0].shape[0]
    atk = df[df['y_test'] == 1].shape[0]
    
    wrong = df[df['y_test'] != df['y_pred']]
    wrong_counts = wrong.groupby('y_test').count()
    
    wrong_normal = wrong_counts.loc[0]['y_pred'] if 0 in wrong_counts.index else 0
    wrong_atk = wrong_counts.loc[1]['y_pred'] if 1 in wrong_counts.index else 0
    
    normal_detect_rate = (normal - wrong_normal) / normal if normal > 0 else 0
    atk_detect_rate = (atk - wrong_atk) / atk if atk > 0 else 0
    
    return normal_detect_rate, atk_detect_rate


def format_3d(df):
    """
    Reshape data in 3D format for RNN/LSTM/GRU inputs.
    
    Args:
        df: Input DataFrame or array
        
    Returns:
        Reshaped array with shape (samples, timesteps, features)
    """
    X = np.array(df)
    return np.reshape(X, (X.shape[0], X.shape[1], 1))


def format_2d(df):
    """
    Reshape data in 2D format for standard ML inputs.
    
    Args:
        df: Input DataFrame or array
        
    Returns:
        Reshaped array with shape (samples, features)
    """
    X = np.array(df)
    return np.reshape(X, (X.shape[0], X.shape[1]))


def compile_train(model, X_train, y_train, deep=True, epochs=10, batch_size=256, 
                  learning_rate=0.001, verbose=1, plot_history=True):
    """
    Unified function to compile and train learning models.
    
    Args:
        model: PyTorch model or scikit-learn model
        X_train: Training features
        y_train: Training labels
        deep: If True, assumes PyTorch deep learning model. If False, assumes scikit-learn
        epochs: Number of training epochs (for deep learning only)
        batch_size: Batch size (for deep learning only)
        learning_rate: Learning rate (for deep learning only)
        verbose: Verbosity level
        plot_history: Whether to plot training history (for deep learning only)
        
    Returns:
        Trained model (and history dict for deep learning models)
    """
    if deep:
        # PyTorch deep learning training
        model = model.to(DEVICE)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
        
        # Convert to tensors if not already
        if not isinstance(X_train, torch.Tensor):
            X_train = torch.FloatTensor(X_train)
            y_train = torch.LongTensor(y_train)
        
        # Create DataLoader
        train_dataset = TensorDataset(X_train, y_train)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        
        history = {'loss': [], 'accuracy': []}
        
        print(f"Training PyTorch model for {epochs} epochs...")
        model.train()
        
        for epoch in range(epochs):
            epoch_loss = 0.0
            correct = 0
            total = 0
            
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                
                epoch_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
            
            epoch_loss /= len(train_loader)
            epoch_acc = correct / total
            
            history['loss'].append(epoch_loss)
            history['accuracy'].append(epoch_acc)
            
            if verbose:
                print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc*100:.2f}%")
        
        # Plot training history
        if plot_history:
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
            
            # Accuracy plot
            ax1.plot(history['accuracy'])
            ax1.set_title('Model Accuracy')
            ax1.set_ylabel('Accuracy')
            ax1.set_xlabel('Epoch')
            ax1.legend(['Train'], loc='upper left')
            ax1.grid(alpha=0.3)
            
            # Loss plot
            ax2.plot(history['loss'])
            ax2.set_title('Model Loss')
            ax2.set_ylabel('Loss')
            ax2.set_xlabel('Epoch')
            ax2.legend(['Train'], loc='upper left')
            ax2.grid(alpha=0.3)
            
            plt.tight_layout()
            plt.savefig(os.path.join(PLOT_DIR, 'training_history.png'), dpi=150)
            plt.close()
        
        print('✅ PyTorch Model Compiled and Trained')
        return model, history
    
    else:
        # Scikit-learn training (SVM, Random Forest, etc.)
        print("Training scikit-learn model...")
        model.fit(X_train, y_train)
        print('✅ Scikit-learn Model Trained')
        return model


def testes(model, X_test, y_test, y_pred=None, deep=True, verbose=1):
    """
    Comprehensive testing function with multiple performance metrics.
    
    Args:
        model: Trained model (PyTorch or scikit-learn)
        X_test: Test features
        y_test: True test labels
        y_pred: Predicted labels (optional, will be generated if None)
        deep: If True, assumes PyTorch model. If False, assumes scikit-learn
        verbose: Verbosity level
        
    Returns:
        Tuple of (accuracy, precision, recall, f1, average)
    """
    # Generate predictions if not provided
    if y_pred is None:
        if deep:
            model = model.to(DEVICE)
            model.eval()
            
            if not isinstance(X_test, torch.Tensor):
                X_test = torch.FloatTensor(X_test)
            
            with torch.no_grad():
                X_test = X_test.to(DEVICE)
                outputs = model(X_test)
                
                # Evaluate model (calculate loss if possible)
                if isinstance(y_test, torch.Tensor):
                    y_test_tensor = y_test.to(DEVICE)
                    criterion = nn.CrossEntropyLoss()
                    loss = criterion(outputs, y_test_tensor)
                    if verbose:
                        print(f"Test Loss: {loss.item():.4f}")
                
                _, y_pred = outputs.max(1)
                y_pred = y_pred.cpu().numpy()
        else:
            # Scikit-learn prediction
            y_pred = model.predict(X_test)
    
    # Ensure y_test is numpy array
    if isinstance(y_test, torch.Tensor):
        y_test = y_test.cpu().numpy()
    
    # Calculate metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    # Calculate average
    avrg = (acc + prec + rec + f1) / 4
    
    if verbose:
        print('\n' + '='*80)
        print('TEST RESULTS')
        print('='*80)
        print(f'Accuracy:  {acc*100:>6.2f}%')
        print(f'Precision: {prec*100:>6.2f}%')
        print(f'Recall:    {rec*100:>6.2f}%')
        print(f'F1 Score:  {f1*100:>6.2f}%')
        print(f'Average:   {avrg*100:>6.2f}%')
        print('='*80)
        
        # Confusion matrix
        cm = confusion_matrix(y_test, y_pred)
        print('\nConfusion Matrix:')
        print(f"           Predicted")
        print(f"           Benign  Attack")
        print(f"Actual Benign  {cm[0,0]:>6,}  {cm[0,1]:>6,}")
        print(f"       Attack  {cm[1,0]:>6,}  {cm[1,1]:>6,}")
        
        # Classification report
        print('\nClassification Report:')
        print(classification_report(y_test, y_pred, target_names=['Benign', 'Attack'], digits=4))
    
    return acc, prec, rec, f1, avrg


def save_model_pytorch(model, name, save_dir=OUT_DIR):
    """
    Save PyTorch model.
    
    Args:
        model: PyTorch model to save
        name: Model name (without extension)
        save_dir: Directory to save model
    """
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f'{name}.pt')
    torch.save(model.state_dict(), save_path)
    print(f'✅ PyTorch model saved to: {save_path}')


def load_model_pytorch(model_class, model_args, name, save_dir=OUT_DIR):
    """
    Load PyTorch model.
    
    Args:
        model_class: Model class (e.g., MLPDropout, DeepCNN)
        model_args: Dictionary of arguments for model initialization
        name: Model name (without extension)
        save_dir: Directory where model is saved
        
    Returns:
        Loaded PyTorch model
    """
    load_path = os.path.join(save_dir, f'{name}.pt')
    model = model_class(**model_args)
    model.load_state_dict(torch.load(load_path, map_location=DEVICE))
    model = model.to(DEVICE)
    model.eval()
    print(f'✅ PyTorch model loaded from: {load_path}')
    return model


def save_Sklearn(model, name, save_dir=OUT_DIR):
    """
    Save scikit-learn model using pickle.
    
    Args:
        model: Scikit-learn model to save
        name: Model name (without extension)
        save_dir: Directory to save model
    """
    import pickle
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f'{name}.pkl')
    with open(save_path, 'wb') as file:
        pickle.dump(model, file)
    print(f'✅ Scikit-learn model saved to: {save_path}')


def load_Sklearn(name, save_dir=OUT_DIR):
    """
    Load scikit-learn model using pickle.
    
    Args:
        name: Model name (without extension)
        save_dir: Directory where model is saved
        
    Returns:
        Loaded scikit-learn model
    """
    import pickle
    load_path = os.path.join(save_dir, f'{name}.pkl')
    with open(load_path, 'rb') as file:
        model = pickle.load(file)
    print(f'✅ Scikit-learn model loaded from: {load_path}')
    return model


# =============================================================================
# MODEL SELECTION - Comment/uncomment to choose which models to train
# =============================================================================
MODELS_TO_TRAIN = {
    'mlp_model': True,          # ✅ Deep MLP (baseline) - Fast, ~5-8 min
    'cnn_model': True,          # ✅ 1D CNN - Moderate, ~8-12 min
    'tcn_model': True,          # ✅ Temporal CNN - Moderate, ~10-15 min
    'bilstm_model': True,       # ✅ BiLSTM-Attention - Moderate, ~10-15 min
}


# =============================================================================
# MODEL ARCHITECTURES (from ids.py)
# =============================================================================

class MLPDropout(nn.Module):
    """Deep MLP with dropout (baseline model)."""
    def __init__(self, d_in: int, d_out: int = 2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.05),
            nn.Linear(64, d_out)
        )
    
    def forward(self, x):
        return self.net(x)


class DeepCNN(nn.Module):
    """1D CNN for feature extraction."""
    def __init__(self, d_in: int, d_out: int = 2):
        super().__init__()
        # Project input to suitable dimension for conv layers
        self.proj = nn.Linear(d_in, 128)
        self.conv1 = nn.Conv1d(1, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.conv3 = nn.Conv1d(128, 64, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(64, 128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, d_out)
        )
    
    def forward(self, x):
        # x: (B, D) → (B, 128)
        x = F.relu(self.proj(x))
        # → (B, 1, 128) for conv
        x = x.unsqueeze(1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = self.pool(x).squeeze(-1)  # (B, 64)
        return self.fc(x)


class DeepTCN(nn.Module):
    """Temporal Convolutional Network."""
    def __init__(self, d_in: int, d_out: int = 2):
        super().__init__()
        self.input = nn.Linear(d_in, 64)
        # Dilated causal convolutions
        self.tcn1 = nn.Conv1d(1, 32, kernel_size=3, dilation=1, padding=1)
        self.tcn2 = nn.Conv1d(32, 64, kernel_size=3, dilation=2, padding=2)
        self.tcn3 = nn.Conv1d(64, 32, kernel_size=3, dilation=4, padding=4)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(32, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, d_out)
        )
    
    def forward(self, x):
        # x: (B, D) → (B, 64)
        x = F.relu(self.input(x))
        # → (B, 1, 64) for conv
        x = x.unsqueeze(1)
        x = F.relu(self.tcn1(x))
        x = F.relu(self.tcn2(x))
        x = F.relu(self.tcn3(x))
        x = self.pool(x).squeeze(-1)  # (B, 32)
        return self.fc(x)


class BiLSTMAttention(nn.Module):
    """Bidirectional LSTM with attention mechanism for IDS classification.
    
    References:
    - IntrusionX (Oct 2025): CNN-LSTM hybrid, 98% accuracy on NSL-KDD (arXiv:2510.00572)
    - xIDS-EnsembleGuard (Mar 2025): LSTM+GRU ensemble on CIC-IDS2017 (arXiv:2503.00615)
    - LSTM-CNN-Attention (Jan 2025): 99.04% accuracy on Edge-IIoTset (arXiv:2501.13962)
    - BiGRU-LSTM-Attention (Aug 2025): Cross-domain IoT security (arXiv:2508.12470)
    
    Architecture:
    - Input projection to reduce dimensionality
    - Bidirectional LSTM for temporal modeling (captures both past and future context)
    - Multi-head attention mechanism to focus on important features/timesteps
    - Dropout regularization to prevent overfitting
    - Output classification layer
    """
    def __init__(self, d_in: int, n_classes: int = 2, p: float = 0.2, hidden_size: int = 128, num_layers: int = 2, num_heads: int = 4):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # Input projection (reduce dimensionality for efficiency)
        self.input_proj = nn.Sequential(
            nn.Linear(d_in, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.Dropout(p)
        )
        
        # Bidirectional LSTM (captures temporal dependencies in both directions)
        self.lstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=p if num_layers > 1 else 0.0,
            bidirectional=True
        )
        
        # Multi-head attention mechanism (focuses on important timesteps/features)
        # Input to attention: hidden_size * 2 (bidirectional concatenation)
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size * 2,
            num_heads=num_heads,
            dropout=p,
            batch_first=True
        )
        
        # Layer normalization after attention
        self.attn_norm = nn.LayerNorm(hidden_size * 2)
        
        # Output classification layers
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(hidden_size, n_classes)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor of shape (batch, features)
        
        Returns:
            logits: Output tensor of shape (batch, n_classes)
        """
        batch_size = x.size(0)
        
        # Project input to hidden dimension
        x = self.input_proj(x)  # (batch, hidden_size)
        
        # Reshape for LSTM: (batch, seq_len=1, hidden_size)
        # For tabular data, we treat each sample as a single timestep
        # The BiLSTM will still learn directional feature dependencies
        x = x.unsqueeze(1)  # (batch, 1, hidden_size)
        
        # Bidirectional LSTM
        # lstm_out: (batch, 1, hidden_size * 2) - concatenated forward and backward
        lstm_out, (h_n, c_n) = self.lstm(x)
        
        # Apply self-attention
        # Query, Key, Value are all from lstm_out
        attn_out, attn_weights = self.attention(
            lstm_out, lstm_out, lstm_out
        )
        
        # Residual connection + layer normalization
        attn_out = self.attn_norm(lstm_out + attn_out)
        
        # Squeeze sequence dimension and classify
        attn_out = attn_out.squeeze(1)  # (batch, hidden_size * 2)
        
        # Classification
        logits = self.classifier(attn_out)  # (batch, n_classes)
        
        return logits


# =============================================================================
# DATA LOADING
# =============================================================================
print("\n" + "="*80)
print("DATA LOADING - CIC-DDoS2019")
print("="*80)

# Load first file - returns (flows_ok, flows_ddos)
flows_ok, flows_ddos = load_huge_file('/kaggle/input/cic-ddos2019-30gb-full-dataset-csv-files/01-12/TFTP.csv')
print('file 1 loaded')

# List of remaining files
files = [
    "DrDoS_LDAP.csv", "DrDoS_MSSQL.csv", "DrDoS_NetBIOS.csv",
    "DrDoS_NTP.csv", "DrDoS_SNMP.csv", "DrDoS_SSDP.csv",
    "DrDoS_UDP.csv", "Syn.csv", "DrDoS_DNS.csv", "UDPLag.csv"
]

# Process each file - keep benign and attack separate across all files
for i, file in enumerate(files, start=2):
    a, b = load_file(f'/kaggle/input/cic-ddos2019-30gb-full-dataset-csv-files/01-12/{file}')
    
    # Append benign flows separately
    flows_ok = pd.concat([flows_ok, a], ignore_index=True)
    # Append attack flows separately
    flows_ddos = pd.concat([flows_ddos, b], ignore_index=True)
    
    print(f'file {i} loaded')

# NOW concatenate benign and attack together (only once at the end)
samples = pd.concat([flows_ok, flows_ddos], ignore_index=True)

# Preprocess training data
samples = preprocess_dataframe(samples, dataset_type='train')

# Save to CSV
samples.to_csv('/kaggle/working/export_dataframe_proc.csv', index=False, header=True)

# Delete large variables
del flows_ok, flows_ddos, a, b

# =============================================================================
# LOAD TEST DATA (Day 2 - 03-11)
# =============================================================================
print("\n" + "-" * 80)
print("LOADING TEST DATA (Day 2)")
print("-" * 80)

# File paths
base_path = "/kaggle/input/cic-ddos2019-30gb-full-dataset-csv-files/03-11/"
files = ["LDAP.csv", "MSSQL.csv", "NetBIOS.csv", "Portmap.csv", "Syn.csv"]
# Uncomment if fixed
# files += ["UDP.csv", "UDPLag.csv"]

# Load first file - returns (flows_ok, flows_ddos)
flows_ok, flows_ddos = load_file(base_path + files[0])
print('file 1 loaded')

# Load remaining files - keep benign and attack separate
for i, file in enumerate(files[1:], start=2):
    a, b = load_file(base_path + file)
    
    # Append benign flows separately
    flows_ok = pd.concat([flows_ok, a], ignore_index=True)
    # Append attack flows separately
    flows_ddos = pd.concat([flows_ddos, b], ignore_index=True)
    
    print(f'file {i} loaded')

# NOW concatenate benign and attack together (only once at the end)
tests = pd.concat([flows_ok, flows_ddos], ignore_index=True)

# Save raw test data to CSV
tests.to_csv('/kaggle/working/export_tests.csv', index=False, header=True)

# Preprocess test data
tests = preprocess_dataframe(tests, dataset_type='test')

# Save preprocessed test data to CSV
tests.to_csv('/kaggle/working/export_tests_proc.csv', index=False, header=True)

# Free memory
del flows_ok, flows_ddos, a, b

print("="*80 + "\n")


# =============================================================================
# DATA PREPARATION FOR TRAINING
# =============================================================================
print("\n" + "="*80)
print("DATA PREPARATION")
print("="*80)

# Use Day 1 (01-12) for training (100% - no validation split)
# Use Day 2 (03-11) for testing (temporal evaluation)
print("\n⚙️ Preparing training data (Day 1 - 01-12)...")
X_train = samples.iloc[:, :-1]  # All features
y_train = samples.iloc[:, -1]   # Labels
print(f"   Training set: {len(X_train):,} samples")

# UPSAMPLE normal flows in training set to balance classes
print("\n⚙️ Upsampling benign flows in training set...")
X_combined = pd.concat([X_train, y_train], axis=1)

# Separate benign and attack flows
is_benign = X_combined[' Label'] == 0
normal = X_combined[is_benign]
ddos = X_combined[~is_benign]

print(f"   Before upsampling - Benign: {len(normal):,}, Attack: {len(ddos):,}")

# Upsample benign to match attack count
normal_upsampled = resample(
    normal,
    replace=True,
    n_samples=len(ddos),
    random_state=SEED
)

# Combine upsampled benign with attacks
upsampled = pd.concat([normal_upsampled, ddos])

# Split back into features and labels
X_train = upsampled.iloc[:, :-1]
y_train = upsampled.iloc[:, -1]

print(f"   After upsampling - Benign: {len(normal_upsampled):,}, Attack: {len(ddos):,}")
print(f"   Final training set: {len(X_train):,} samples")

# Free memory
del X_combined, normal, ddos, normal_upsampled, upsampled

# Prepare test data (Day 2)
print("\n⚙️ Preparing test data (Day 2)...")
X_test = tests.iloc[:, :-1]
y_test = tests.iloc[:, -1]
print(f"   Test set: {len(X_test):,} samples")

# Normalize data
print("\n⚙️ Normalizing features (MinMaxScaler -1 to 1)...")
X_train_np, X_test_np = normalize_data(X_train.values, X_test.values)

print(f"   Feature range: [{X_train_np.min():.2f}, {X_train_np.max():.2f}]")

# Get input dimension
input_size = X_train_np.shape[1]
print(f"\n📊 Input dimension: {input_size} features")

# Convert to PyTorch tensors
print("\n⚙️ Converting to PyTorch tensors...")
X_train_tensor = torch.FloatTensor(X_train_np)
y_train_tensor = torch.LongTensor(y_train.values)
X_test_tensor = torch.FloatTensor(X_test_np)
y_test_tensor = torch.LongTensor(y_test.values)

# Create DataLoaders
batch_size = 256
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4 if not USE_XLA else 0, pin_memory=True if not USE_XLA else False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4 if not USE_XLA else 0, pin_memory=True if not USE_XLA else False)

# Wrap with TPU ParallelLoader for multi-core data distribution
if USE_XLA and xm is not None:
    import torch_xla.distributed.parallel_loader as pl
    train_loader = pl.MpDeviceLoader(train_loader, DEVICE)
    test_loader = pl.MpDeviceLoader(test_loader, DEVICE)
    print("✅ TPU ParallelLoader enabled for efficient multi-core training")

print(f"   Batch size: {batch_size}")
print(f"   Train batches: {len(train_loader)}")
print(f"   Test batches: {len(test_loader)}")

# Class distribution
print("\n📊 Class Distribution:")
print(f"   Training (Day 1) - Benign: {(y_train==0).sum():,}, Attack: {(y_train==1).sum():,}")
print(f"   Test (Day 2)     - Benign: {(y_test==0).sum():,}, Attack: {(y_test==1).sum():,}")

print("="*80 + "\n")


# =============================================================================
# TRAINING UTILITIES
# =============================================================================

def train_model(model, train_loader, epochs=10, learning_rate=0.001, model_name="Model"):
    """Train a PyTorch model with TPU/GPU optimization.
    
    Note: No validation set - using temporal evaluation (Day 1 train → Day 2 test).
    """
    print(f"\n{'='*80}")
    print(f"TRAINING: {model_name}")
    print(f"{'='*80}")
    print(f"Device: {DEVICE}, TPU: {USE_XLA}")
    print(f"Temporal Split: Day 1 (train) → Day 2 (test)")
    
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    
    train_losses = []
    train_accs = []
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        correct = 0
        total = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", disable=USE_XLA)  # Disable tqdm for TPU
        for batch_idx, (inputs, labels) in enumerate(pbar if not USE_XLA else train_loader):
            if not USE_XLA:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            # TPU: Mark step for graph execution synchronization
            if USE_XLA and xm is not None:
                xm.mark_step()
            
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            if not USE_XLA:
                pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100.*correct/total:.2f}%'})
            elif batch_idx % 50 == 0:  # Print every 50 batches for TPU
                print(f"  Batch {batch_idx}/{len(train_loader)} - Loss: {loss.item():.4f}, Acc: {100.*correct/total:.2f}%")
        
        train_loss /= len(train_loader)
        train_acc = correct / total
        
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        
        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc*100:.2f}%")
    
    print(f"\n✅ Training complete!")
    
    return model, {'train_losses': train_losses, 'train_accs': train_accs}


def evaluate_model(model, test_loader, model_name="Model"):
    """Evaluate a PyTorch model on test set."""
    print(f"\n{'='*80}")
    print(f"EVALUATION: {model_name}")
    print(f"{'='*80}")
    
    model = model.to(DEVICE)
    model.eval()
    
    all_preds = []
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="Evaluating", disable=USE_XLA):
            if not USE_XLA:
                inputs = inputs.to(DEVICE)
            outputs = model(inputs)
            probs = F.softmax(outputs, dim=1)[:, 1]
            _, predicted = outputs.max(1)
            
            # TPU: Mark step for evaluation
            if USE_XLA and xm is not None:
                xm.mark_step()
            
            all_preds.extend(predicted.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy() if USE_XLA else labels.numpy())
    
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    
    # Calculate metrics
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, zero_division=0)
    rec = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    auc_score = roc_auc_score(all_labels, all_probs)
    
    # Detection rates
    norm_detect, atk_detect = test_normal_atk(all_labels, all_preds)
    
    # False positive rate
    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
    fp_rate = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    print(f"\n📊 Test Set Evaluation:")
    print("-" * 80)
    print(f"    Accuracy:           {acc*100:>6.2f}%")
    print(f"    Precision:          {prec*100:>6.2f}%")
    print(f"    Recall:             {rec*100:>6.2f}%")
    print(f"    F1 Score:           {f1*100:>6.2f}%")
    print(f"    AUC-ROC:            {auc_score:>8.4f}")
    print(f"    Normal Detect Rate: {norm_detect*100:>6.2f}%")
    print(f"    Attack Detect Rate: {atk_detect*100:>6.2f}%")
    print(f"    False Positive Rate:{fp_rate*100:>6.2f}%")
    print("\n    Confusion Matrix:")
    cm = confusion_matrix(all_labels, all_preds)
    print(f"                Predicted")
    print(f"                Benign  Attack")
    print(f"    Actual Benign  {cm[0,0]:>6,}  {cm[0,1]:>6,}")
    print(f"           Attack  {cm[1,0]:>6,}  {cm[1,1]:>6,}")
    print("\n    Classification Report:")
    print(classification_report(all_labels, all_preds, target_names=['Benign', 'Attack'], digits=4))
    
    return {
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'auc': auc_score,
        'normal_detect_rate': norm_detect,
        'attack_detect_rate': atk_detect,
        'false_positive_rate': fp_rate,
        'predictions': all_preds,
        'probabilities': all_probs,
        'labels': all_labels,
        'confusion_matrix': cm
    }


# =============================================================================
# MODEL TRAINING
# =============================================================================
print("\n" + "="*80)
print("MODEL TRAINING")
print("="*80)

trained_models = {}
results = {}

# Train MLP
if MODELS_TO_TRAIN.get('mlp_model', False):
    mlp_model = MLPDropout(d_in=input_size, d_out=2)
    mlp_model, mlp_history = train_model(mlp_model, train_loader, epochs=10, model_name="MLP")
    trained_models['mlp'] = mlp_model
    results['mlp'] = evaluate_model(mlp_model, test_loader, model_name="MLP")
    
    # Save model
    torch.save(mlp_model.state_dict(), os.path.join(OUT_DIR, 'mlp_model.pt'))

# Train CNN
if MODELS_TO_TRAIN.get('cnn_model', False):
    cnn_model = DeepCNN(d_in=input_size, d_out=2)
    cnn_model, cnn_history = train_model(cnn_model, train_loader, epochs=10, model_name="CNN")
    trained_models['cnn'] = cnn_model
    results['cnn'] = evaluate_model(cnn_model, test_loader, model_name="CNN")
    
    # Save model
    torch.save(cnn_model.state_dict(), os.path.join(OUT_DIR, 'cnn_model.pt'))

# Train TCN
if MODELS_TO_TRAIN.get('tcn_model', False):
    tcn_model = DeepTCN(d_in=input_size, d_out=2)
    tcn_model, tcn_history = train_model(tcn_model, train_loader, epochs=10, model_name="TCN")
    trained_models['tcn'] = tcn_model
    results['tcn'] = evaluate_model(tcn_model, test_loader, model_name="TCN")
    
    # Save model
    torch.save(tcn_model.state_dict(), os.path.join(OUT_DIR, 'tcn_model.pt'))

# Train BiLSTM-Attention
if MODELS_TO_TRAIN.get('bilstm_model', False):
    bilstm_model = BiLSTMAttention(d_in=input_size, n_classes=2)
    bilstm_model, bilstm_history = train_model(bilstm_model, train_loader, epochs=10, model_name="BiLSTM-Attention")
    trained_models['bilstm'] = bilstm_model
    results['bilstm'] = evaluate_model(bilstm_model, test_loader, model_name="BiLSTM-Attention")
    
    # Save model
    torch.save(bilstm_model.state_dict(), os.path.join(OUT_DIR, 'bilstm_model.pt'))

# Find best model
best_f1 = 0.0
best_model_name = None
for name, result in results.items():
    if result['f1'] > best_f1:
        best_f1 = result['f1']
        best_model_name = name

print(f"\n🏆 Best model: {best_model_name.upper()} (F1: {best_f1*100:.2f}%)")


# =============================================================================
# PLOTTING FUNCTIONS
# =============================================================================

def plot_confusion_matrix(cm, model_name, save_path):
    """Plot confusion matrix heatmap."""
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Benign', 'Attack'], 
                yticklabels=['Benign', 'Attack'])
    plt.title(f'Confusion Matrix - {model_name}')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


def plot_roc_curve(labels, probs, model_name, save_path):
    """Plot ROC curve."""
    fpr, tpr, _ = roc_curve(labels, probs)
    roc_auc = auc(fpr, tpr)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f'{model_name} (AUC = {roc_auc:.4f})', linewidth=2)
    plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve - {model_name}')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


def plot_pr_curve(labels, probs, model_name, save_path):
    """Plot Precision-Recall curve."""
    precision, recall, _ = precision_recall_curve(labels, probs)
    pr_auc = auc(recall, precision)
    
    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, label=f'{model_name} (AUC = {pr_auc:.4f})', linewidth=2)
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(f'Precision-Recall Curve - {model_name}')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


def plot_metrics_comparison(results, save_path):
    """Plot bar chart comparing model metrics."""
    metrics = ['accuracy', 'precision', 'recall', 'f1', 'auc']
    model_names = list(results.keys())
    
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(metrics))
    width = 0.8 / len(model_names)
    
    for i, model_name in enumerate(model_names):
        values = [results[model_name][m] * 100 for m in metrics]
        ax.bar(x + i * width, values, width, label=model_name.upper())
    
    ax.set_xlabel('Metrics')
    ax.set_ylabel('Score (%)')
    ax.set_title('Model Performance Comparison')
    ax.set_xticks(x + width * (len(model_names) - 1) / 2)
    ax.set_xticklabels([m.upper() for m in metrics])
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


# Generate plots for each model
print("\n" + "="*80)
print("GENERATING PLOTS")
print("="*80)

for name, result in results.items():
    print(f"\nGenerating plots for {name.upper()}...")
    
    plot_confusion_matrix(result['confusion_matrix'], name.upper(), 
                         os.path.join(PLOT_DIR, f'{name}_confusion_matrix.png'))
    plot_roc_curve(result['labels'], result['probabilities'], name.upper(), 
                  os.path.join(PLOT_DIR, f'{name}_roc_curve.png'))
    plot_pr_curve(result['labels'], result['probabilities'], name.upper(), 
                 os.path.join(PLOT_DIR, f'{name}_pr_curve.png'))

# Comparison plot
if len(results) > 1:
    plot_metrics_comparison(results, os.path.join(PLOT_DIR, 'model_comparison.png'))

print("\n✅ All plots saved to:", PLOT_DIR)
print("="*80 + "\n")


# =============================================================================
# ENSEMBLE EVALUATION (Baseline - Static Mean Logits)
# =============================================================================
if len(trained_models) > 1:
    print("\n" + "="*80)
    print("ENSEMBLE EVALUATION (BASELINE)")
    print("="*80)
    print("Strategy: Static Mean Logits (all models weighted equally)")
    print(f"Models in ensemble: {list(trained_models.keys())}")
    print("-" * 80)
    
    # Evaluate ensemble
    all_preds = []
    all_probs = []
    all_labels = []
    
    for model in trained_models.values():
        model.eval()
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            if not USE_XLA:
                inputs = inputs.to(DEVICE)
            
            # Collect logits from all models
            logits_list = []
            for model in trained_models.values():
                logits = model(inputs)
                logits_list.append(logits)
            
            # Mean logits ensemble
            ensemble_logits = torch.stack(logits_list, dim=0).mean(dim=0)
            probs = F.softmax(ensemble_logits, dim=1)[:, 1]
            _, predicted = torch.max(ensemble_logits.data, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy() if USE_XLA else labels.numpy())
    
    # Calculate metrics
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    
    ensemble_accuracy = accuracy_score(all_labels, all_preds)
    ensemble_precision = precision_score(all_labels, all_preds, zero_division=0)
    ensemble_recall = recall_score(all_labels, all_preds, zero_division=0)
    ensemble_f1 = f1_score(all_labels, all_preds, zero_division=0)
    ensemble_auc = roc_auc_score(all_labels, all_probs)
    
    print(f"\n📊 Ensemble Test Set Evaluation:")
    print("-" * 80)
    print(f"    Accuracy:  {ensemble_accuracy*100:>6.2f}%")
    print(f"    Precision: {ensemble_precision*100:>6.2f}%")
    print(f"    Recall:    {ensemble_recall*100:>6.2f}%")
    print(f"    F1 Score:  {ensemble_f1*100:>6.2f}%")
    print(f"    AUC-ROC:   {ensemble_auc:>8.4f}")
    print("\n    Confusion Matrix:")
    ensemble_cm = confusion_matrix(all_labels, all_preds)
    print(f"                Predicted")
    print(f"                Benign  Attack")
    print(f"    Actual Benign  {ensemble_cm[0,0]:>6,}  {ensemble_cm[0,1]:>6,}")
    print(f"           Attack  {ensemble_cm[1,0]:>6,}  {ensemble_cm[1,1]:>6,}")
    print("\n    Classification Report:")
    print(classification_report(all_labels, all_preds, 
                                target_names=['Benign', 'Attack'], digits=4))
    
    # Add to results for comparison
    results['ensemble'] = {
        'accuracy': ensemble_accuracy,
        'precision': ensemble_precision,
        'recall': ensemble_recall,
        'f1': ensemble_f1,
        'auc': ensemble_auc,
        'predictions': all_preds,
        'probabilities': all_probs,
        'labels': all_labels
    }
    
    # Generate ensemble plots
    plot_confusion_matrix(ensemble_cm, 'Ensemble (Mean Logits)', 
                         os.path.join(PLOT_DIR, 'ensemble_confusion_matrix.png'))
    plot_roc_curve(all_labels, all_probs, 'Ensemble (Mean Logits)', 
                  os.path.join(PLOT_DIR, 'ensemble_roc_curve.png'))
    plot_pr_curve(all_labels, all_probs, 'Ensemble (Mean Logits)', 
                 os.path.join(PLOT_DIR, 'ensemble_pr_curve.png'))
    
    # Update comparison plots with ensemble
    plot_metrics_comparison(results, os.path.join(PLOT_DIR, 'model_comparison_with_ensemble.png'))
    
    # Print improvement over best single model
    if best_f1 < ensemble_f1:
        improvement = (ensemble_f1 - best_f1) * 100
        print(f"\n🎯 Ensemble improvement: +{improvement:.2f}% F1 over best single model")
    else:
        degradation = (best_f1 - ensemble_f1) * 100
        print(f"\n⚠️ Ensemble degradation: -{degradation:.2f}% F1 vs best single model")

print("\n" + "="*80)
print("TRAINING COMPLETE")
print("="*80 + "\n")

# =============================================================================
# FINAL RESULTS TABLE
# =============================================================================
print("\n" + "="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)
print(f"\n{'Model':<15} {'Accuracy':<10} {'Precision':<11} {'Recall':<10} {'F1 Score':<10} {'AUC-ROC':<10} {'Normal Det':<11} {'Attack Det':<11} {'False Pos':<10}")
print("-" * 80)

for model_name, result in results.items():
    norm_det, atk_det = test_normal_atk(result['labels'], result['predictions'])
    tn, fp, fn, tp = confusion_matrix(result['labels'], result['predictions']).ravel()
    fp_rate = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    print(f"{model_name.upper():<15} {result['accuracy']*100:>8.2f}%  {result['precision']*100:>9.2f}%  {result['recall']*100:>8.2f}%  {result['f1']*100:>8.2f}%  {result['auc']:>8.4f}  {norm_det*100:>9.2f}%  {atk_det*100:>9.2f}%  {fp_rate*100:>8.2f}%")

print("="*80 + "\n")







HARDWARE DETECTION
PyTorch: 2.8.0+cpu
Platform: Linux-6.1.42+-x86_64-with-glibc2.41


/usr/local/lib/python3.12/site-packages/torch_xla/__init__.py:258: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(


✅ TPU Available: torch_xla 2.8.0

⚠️ CUDA not available



E0000 00:00:1763538413.867328      74 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:238


🎯 Using TPU: xla:0


DATA LOADING - CIC-DDoS2019


...


Generating plots for TCN...



Generating plots for BILSTM...



✅ All plots saved to: /kaggle/working/plots


ENSEMBLE EVALUATION (BASELINE)
Strategy: Static Mean Logits (all models weighted equally)
Models in ensemble: ['mlp', 'cnn', 'tcn', 'bilstm']
--------------------------------------------------------------------------------



📊 Ensemble Test Set Evaluation:
--------------------------------------------------------------------------------
    Accuracy:   99.93%
    Precision:  99.95%
    Recall:     99.97%
    F1 Score:   99.96%
    AUC-ROC:     1.0000

    Confusion Matrix:
                Predicted
                Benign  Attack
    Actual Benign  49,636     127
           Attack      85  248,730

    Classification Report:
              precision    recall  f1-score   support

      Benign     0.9983    0.9974    0.9979     49763
      Attack     0.9995    0.9997    0.9996    248815

    accuracy                         0.9993    298578
   macro avg     0.9989    0.9986    0.9987    298578
weighted avg     0.9993    0.9993    0.9993    298578




⚠️ Ensemble degradation: -0.01% F1 vs best single model

TRAINING COMPLETE


FINAL RESULTS SUMMARY

Model           Accuracy   Precision   Recall     F1 Score   AUC-ROC    Normal Det  Attack Det  False Pos 
--------------------------------------------------------------------------------
MLP                99.93%      99.93%     99.98%     99.96%    1.0000      99.65%      99.98%      0.35%
CNN                99.94%      99.94%     99.99%     99.96%    0.9999      99.71%      99.99%      0.29%
TCN                99.22%      99.99%     99.08%     99.53%    1.0000      99.95%      99.08%      0.05%
BILSTM             99.69%      99.97%     99.66%     99.81%    1.0000      99.83%      99.66%      0.17%
ENSEMBLE           99.93%      99.95%     99.97%     99.96%    1.0000      99.74%      99.97%      0.26%

